# Sensitivity of Reserves to Individual Link Ratios

In [1]:
import numpy as np
import pandas as pd

import chainladder as cl

`Development(drop=(origin, age))` leaves one link ratio out of the average.
Dropping each observed ratio in turn and refitting shows which ratios move
the selected factors and the reserve the most. Verdonck and Debruyne show
that the influence of a single cell on the chain-ladder estimates is
unbounded {cite}`verdonck2011`, so a large move here marks a ratio worth a
closer look.

`GridSearch` already refits an estimator once per parameter value.
`GridSearch.leave_one_out_grid` builds the parameter grid. It returns `None`
for the base fit and one `(origin, age)` pair for every observed link ratio,
including ages that have only one ratio. `Development` drops the ratio at
that origin and age.

In [2]:
raa = cl.load_sample("raa")
raa_grid = cl.GridSearch.leave_one_out_grid(raa)
raa_grid["dev__drop"][:3]

[None, ('1981', 12), ('1981', 24)]

The grid key `dev__drop` sends each value to the `drop` parameter of the
step named `dev` in a `Pipeline`. The built-in scorer `GridSearch.ibnr`
records total IBNR, and `GridSearch.ldf` records the `ldf_` of the final step
by age.

In [3]:
pipe = cl.Pipeline([("dev", cl.Development()), ("model", cl.Chainladder())])
scoring = {"ibnr": cl.GridSearch.ibnr, "ldf": cl.GridSearch.ldf}
raa_search = cl.GridSearch(pipe, raa_grid, scoring).fit(raa)
len(raa_search.results_)

46

That is the base fit plus 45 refits, one for each link ratio in the 10 by 10
triangle. `compare_results` compares every refit with the base fit and
returns one row per dropped ratio. `GridSearch.leave_one_out_grid_labels`
names each row by the dropped ratio. The comparisons in
`GridSearch.comparisons` add the change in total IBNR, its share of the base,
and the factor at that age in the base fit and in the scenario that drops the
ratio. A `DataFrame.apply` over the rows looks up each dropped ratio in the
triangle and adds it as `link_ratio`. `compare_results` keeps the order of
the grid, so the table below sorts the rows by the size of the IBNR change.

In [4]:
def with_link_ratios(table, triangle):
    ratios = triangle.link_ratio.to_frame(origin_as_datetime=False)
    ratios.index = ratios.index.astype(str)
    ratios.columns = [int(column.split("-")[0]) for column in ratios.columns]
    table = table.copy()
    table.insert(
        2,
        "link_ratio",
        table.apply(lambda row: ratios.at[row["origin"], row["age"]], axis=1),
    )
    return table


formats = {
    "link_ratio": "{:.4f}",
    "ibnr_change": "{:,.0f}",
    "ibnr_share": "{:.1%}",
    "ldf_base": "{:.4f}",
    "ldf_scenario": "{:.4f}",
}
compare = {
    "base": "base",
    "result_labels": cl.GridSearch.leave_one_out_grid_labels,
    "result_comparisons": {
        "ibnr": [cl.GridSearch.comparisons.change, cl.GridSearch.comparisons.share],
        "ldf": [cl.GridSearch.comparisons.base, cl.GridSearch.comparisons.scenario],
    },
}
raa_table = with_link_ratios(raa_search.compare_results(**compare), raa).sort_values(
    "ibnr_change", key=abs, ascending=False
)
raa_table.head(5).style.format(formats, na_rep="NaN").hide(axis="index")

origin,age,link_ratio,ibnr_change,ibnr_share,ldf_base,ldf_scenario
1985,60,1.0087,"3,363",6.5%,1.1134,1.1525
1981,96,1.0029,"2,818",5.4%,1.0169,1.0331
1981,12,1.6498,"2,468",4.7%,2.9994,3.4016
1982,96,1.0331,"-2,449",-4.7%,1.0169,1.0029
1982,36,1.9766,"-2,217",-4.3%,1.2709,1.2227


On `raa`, base IBNR is 52,135 and no single ratio moves it by more than
6.5%. Two of the 45 ratios move it by more than 5%. The largest move comes
from 1985 at 60, an ordinary looking ratio of 1.0087. It is one of only five
ratios at 60-72, so dropping it raises that factor from 1.1134 to 1.1525 and
IBNR by 3,363.

In [5]:
genins = cl.load_sample("genins")
genins_grid = cl.GridSearch.leave_one_out_grid(genins)
genins_search = cl.GridSearch(pipe, genins_grid, scoring).fit(genins)
genins_table = with_link_ratios(
    genins_search.compare_results(**compare), genins
).sort_values("ibnr_change", key=abs, ascending=False)
genins_table.head(5).style.format(formats, na_rep="NaN").hide(axis="index")

origin,age,link_ratio,ibnr_change,ibnr_share,ldf_base,ldf_scenario
2001,108,1.0177,"-855,780",-4.6%,1.0177,NaN
2002,96,1.0865,"-549,914",-2.9%,1.0766,1.0630
2004,36,1.7118,"-518,565",-2.8%,1.4574,1.4140
2004,48,1.0725,"509,117",2.7%,1.1739,1.1998
2003,60,1.0369,"487,330",2.6%,1.1038,1.1229


On `genins`, base IBNR is 18,680,856. The largest move comes from 2001 at
108, and its `ldf_scenario` reads NaN. Both sample triangles have a single
ratio at 108-120, so dropping it leaves nothing to average:

In [6]:
dropped = cl.Development(drop=("2001", 108)).fit(genins)
pd.DataFrame(
    {
        "ldf_": dropped.ldf_.values[0, 0, 0, -2:],
        "cdf_": dropped.cdf_.values[0, 0, 0, -2:],
    },
    index=pd.Index([96, 108], name="age"),
)

,ldf_,cdf_
age,,
96,1.076555,1.076555
108,NaN,1.000000


With no ratio left, `ldf_` at 108 is NaN while `cdf_` at 108 is 1, so
`Chainladder` treats the missing factor as 1 and raises no warning. That row
measures the effect of setting the last factor to 1, which is a different
question from leaving out one ratio among several. The `raa` grid has the
same row at 1981 at 108, where IBNR falls by 1,774, or 3.4%. Among the
ratios that leave a factor behind, the largest move on `genins` comes from
2002 at 96, at -549,914 or -2.9%.

In [7]:
import matplotlib.pyplot as plt

plt.style.use("ggplot")
%config InlineBackend.figure_format = 'retina'

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (name, table) in zip(axes, {"raa": raa_table, "genins": genins_table}.items()):
    shares = table.pivot(index="origin", columns="age", values="ibnr_share")
    limit = np.abs(shares.values[np.isfinite(shares.values)]).max()
    image = ax.imshow(shares.values, cmap="RdBu_r", vmin=-limit, vmax=limit)
    for (i, j), value in np.ndenumerate(shares.values):
        if np.isfinite(value):
            label = f"{value:.1%}".replace("-0.0%", "0.0%")
            color = "white" if abs(value) > 0.6 * limit else "black"
            ax.text(j, i, label, ha="center", va="center", fontsize=6.5, color=color)
    single = table[table["ldf_scenario"].isna()]
    for origin, age in zip(single["origin"], single["age"]):
        i, j = list(shares.index).index(origin), list(shares.columns).index(age)
        ax.add_patch(
            plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="black", lw=2)
        )
    ax.set_xticks(range(len(shares.columns)), shares.columns)
    ax.set_yticks(range(len(shares.index)), shares.index)
    ax.set_xlabel("Age of the dropped link ratio")
    ax.set_ylabel("Origin")
    ax.set_title(f"{name}: change in total IBNR")
    ax.grid(False)
    fig.colorbar(image, ax=ax, format=lambda x, _: f"{x:.0%}", shrink=0.8)
fig.tight_layout()

Each cell shows the change in total IBNR when that link ratio is left out,
and a black box marks the single-ratio cells. On `raa`, 20 of the 45 refits
move IBNR by more than 2%, against 9 of 45 on `genins`. To use the helper
with a pipeline that names its `Development` step differently, pass that
name as `step`. Scoring callables of your own can sit beside the built-in
ones, and `compare_results` keeps their columns from `results_` unless
`result_comparisons` lists comparisons for them. A comparison of your own
takes the same four inputs as the built-in ones.